In [3]:
import os
# Fixer le répertoire de travail sur la racine de ton projet
os.chdir('/Users/7maksacod/Documents/ISM/Master_2/Memoire_big_data_ia')

In [6]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge

# Tenter d'importer LightGBM si disponible
try:
    from lightgbm import LGBMRegressor
    HAS_LIGHTGBM = True
except ImportError:
    HAS_LIGHTGBM = False

warnings.filterwarnings('ignore')

# =====================================================================
# 1. CHARGEMENT ET PRÉTRAITEMENT INTELLIGENT DU DATASET GOLD
# =====================================================================

DATA_PATH = "data/processed/dataset_dakar_gold.csv"
FALLBACK_PATH = "dataset_dakar_gold.csv"

def clean_dataset_intelligent(df):
    initial_len = len(df)
    
    # Suppression de toute colonne fantôme générée par une mauvaise lecture CSV
    cols_bizarres = [c for c in df.columns if ',' in c]
    if cols_bizarres:
        df = df.drop(columns=cols_bizarres)

    # Dédoublonnage sécurisé (on évite de supprimer les lignes dont l'URL est NaN)
    if 'url' in df.columns:
        df_avec_url = df[df['url'].notna()].drop_duplicates(subset=['url'], keep='first')
        df_sans_url = df[df['url'].isna()]
        if 'description_complete' in df_sans_url.columns:
            df_sans_url = df_sans_url.drop_duplicates(subset=['description_complete'], keep='first')
        df = pd.concat([df_avec_url, df_sans_url], ignore_index=True)
    elif 'description_complete' in df.columns:
        df = df.drop_duplicates(subset=['description_complete'], keep='first')
        
    # Filtrage souple des loyers (15 000 FCFA à 10 000 000 FCFA)
    if 'prix_fcfa' in df.columns:
        df = df[df['prix_fcfa'].notnull()]
        df = df[(df['prix_fcfa'] >= 15000) & (df['prix_fcfa'] <= 10000000)]
        
    if 'description_complete' in df.columns:
        df['description_complete'] = df['description_complete'].fillna('')
        
    print(f"🧹 Nettoyage intelligent terminé : {len(df)} annonces conservées sur {initial_len} initiales.")
    return df

# Chargement robuste
if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH, sep=';', on_bad_lines='skip', engine='python')
elif os.path.exists(FALLBACK_PATH):
    df = pd.read_csv(FALLBACK_PATH, sep=';', on_bad_lines='skip', engine='python')
else:
    raise FileNotFoundError("⚠️ Impossible de trouver le fichier 'dataset_dakar_gold.csv'.")

df.columns = df.columns.str.strip().str.lower()
df = clean_dataset_intelligent(df)

df['quartier'] = df['quartier'].fillna('Inconnu')

# =====================================================================
# 2. SÉPARATION DES FEATURES ET TARGET
# =====================================================================

target_col = 'prix_fcfa'

num_features = [
    'nb_pieces', 'nb_salles_bain', 'ratio_bains_chambres',
    'dist_plateau_km', 'dist_vdn_km', 'dist_mer_km',
    'has_groupe', 'has_reserve_eau', 'is_studio', 'is_villa',
    'has_piscine', 'has_ascenseur', 'has_parking', 'has_securite',
    'has_clim', 'has_vue_mer', 'is_meuble', 'has_cuisine_equipee',
    'has_balcon', 'is_agence_pro', 'longueur_description'
]

num_features = [col for col in num_features if col in df.columns]
cat_features = ['quartier']
text_feature = 'description_complete' if 'description_complete' in df.columns else None

features_to_use = num_features + cat_features
if text_feature:
    features_to_use.append(text_feature)

X = df[features_to_use]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

FRENCH_STOP_WORDS = [
    'de', 'la', 'le', 'et', 'à', 'en', 'un', 'une', 'du', 'des', 'les', 'pour',
    'dans', 'avec', 'sur', 'par', 'au', 'aux', 'est', 'sont', 'pas', 'plus',
    'aussi', 'ce', 'cette', 'ces', 'se', 'sa', 'son', 'ses', 'ne', 'ou', 'très'
]

# =====================================================================
# 3. PIPELINE DE PRÉTRAITEMENT ROBUSTE
# =====================================================================

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Inconnu')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

transformers = [
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
]

if text_feature:
    text_transformer = TfidfVectorizer(
        max_features=120, 
        stop_words=FRENCH_STOP_WORDS,
        min_df=2
    )
    transformers.append(('text', text_transformer, text_feature))

preprocessor = ColumnTransformer(transformers=transformers)

# =====================================================================
# 4. BENCHMARK ET ÉVALUATION ($R^2$, MAE, RMSE)
# =====================================================================

models = {
    "Gradient Boosting (Huber Loss)": GradientBoostingRegressor(
        loss='huber', n_estimators=350, learning_rate=0.03, max_depth=5, random_state=42
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=300, max_depth=15, random_state=42
    ),
    "Régression Ridge": Ridge(alpha=3.0)
}

if HAS_LIGHTGBM:
    models["LightGBM"] = LGBMRegressor(
        n_estimators=350, learning_rate=0.03, num_leaves=31, random_state=42, verbose=-1
    )

print("\n🚀 Début de l'entraînement et évaluation du Benchmark...")

results = []
trained_models = {}

for name, base_model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', TransformedTargetRegressor(
            regressor=base_model,
            func=np.log1p,
            inverse_func=np.expm1
        ))
    ])
    
    pipeline.fit(X_train, y_train)
    
    # Prédictions à l'échelle réelle (expm1 appliqué automatiquement)
    y_pred = pipeline.predict(X_test)
    
    # Métriques réelles
    r2_real = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    results.append({
        'Modèle': name,
        'R²': round(r2_real, 4),
        'MAE (FCFA)': round(mae, 1),
        'RMSE (FCFA)': round(rmse, 1)
    })
    
    trained_models[name] = (pipeline, r2_real)

# =====================================================================
# 5. AFFICHAGE ET SAUVEGARDE DU MEILLEUR MODÈLE
# =====================================================================

results_df = pd.DataFrame(results).sort_values(by='R²', ascending=False)

print("\n=======================================================================")
print("🏆 TABLEAU COMPARATIF DES PERFORMANCES (BENCHMARK FINAL)")
print("=======================================================================")
print(results_df.to_string(index=False))
print("=======================================================================")

best_model_name = results_df.iloc[0]['Modèle']
best_pipeline, _ = trained_models[best_model_name]

os.makedirs('models', exist_ok=True)
joblib.dump(best_pipeline, 'models/model_dakar_immo.pkl')

print(f"\n✅ MEILLEUR MODÈLE ENREGISTRÉ : {best_model_name}")
print(f"📁 Modèle sérialisé dans : 'models/model_dakar_immo.pkl'")

🧹 Nettoyage intelligent terminé : 2905 annonces conservées sur 3515 initiales.

🚀 Début de l'entraînement et évaluation du Benchmark...

🏆 TABLEAU COMPARATIF DES PERFORMANCES (BENCHMARK FINAL)
                        Modèle     R²  MAE (FCFA)  RMSE (FCFA)
                      LightGBM 0.6400    164107.9     306395.5
                 Random Forest 0.6202    166894.5     314704.5
Gradient Boosting (Huber Loss) 0.6050    164871.6     320939.3
              Régression Ridge 0.5762    175248.4     332445.8

✅ MEILLEUR MODÈLE ENREGISTRÉ : LightGBM
📁 Modèle sérialisé dans : 'models/model_dakar_immo.pkl'
